# Caso Práctico: _Support Vector Machine (SVM)_

## Conjunto de datos: Detección de URLs maliciosas

### Descripción
Desde hace tiempo, la Web se ha convertido en una plataforma importante para actividades delictivas en línea, y las URL son el principal vehículo en este ámbito. Para contrarrestar estos problemas, la comunidad de seguridad ha centrado sus esfuerzos en desarrollar técnicas orientadas, sobre todo, a incluir las URL maliciosas en listas negras.

Este enfoque protege a los usuarios de los dominios maliciosos ya conocidos, pero solo resuelve una parte del problema. Las nuevas URL maliciosas que surgen masivamente por toda la web suelen tomar la delantera en esta carrera. Además, sitios web confiables y bien posicionados en el ranking de Alexa pueden contener URL fraudulentas comprometidas, conocidas como URL de *defacement* (desfiguración).

Estudiamos principalmente cinco tipos de URL:

- **URL benignas**: Se recopilaron más de 35 300 URL benignas de los sitios web mejor posicionados en Alexa. Los dominios se procesaron con el rastreador web Heritrix para extraer las URL. Inicialmente se rastrearon alrededor de medio millón de URL únicas, que luego se filtraron para eliminar duplicados y URL compuestas solo por el dominio. Después, las URL extraídas se verificaron en VirusTotal para quedarse únicamente con las benignas.
- **URL de spam**: Se recopilaron alrededor de 12 000 URL de spam del conjunto de datos público WEBSPAM-UK2007.
- **URL de phishing**: Se obtuvieron alrededor de 10 000 URL de phishing de OpenPhish, un repositorio de sitios de phishing activos.
- **URL de malware**: Se obtuvieron más de 11 500 URL relacionadas con sitios de malware de DNS-BH, un proyecto que mantiene una lista de sitios de malware.
- **URL de defacement**: Más de 45 450 URL pertenecen a la categoría de *defacement*. Corresponden a sitios web confiables del ranking de Alexa que alojan URL fraudulentas u ocultas que llevan a páginas web maliciosas.

### Descarga de los ficheros de datos
[https://www.unb.ca/cic/datasets/url-2016.html](https://www.unb.ca/cic/datasets/url-2016.html)

### Referencias adicionales sobre el conjunto de datos
_Mohammad Saiful Islam Mamun, Mohammad Ahmad Rathore, Arash Habibi Lashkari, Natalia Stakhanova y Ali A. Ghorbani, "Detecting Malicious URLs Using Lexical Analysis", Network and System Security, Springer International Publishing, pp. 467–482, 2016._

## Imports

In [10]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline

**Interpretacion:**
* Es la implementacion de librerias tanto de numpy y pandas, matplotlib para dibujar las gráficas, y las funciones de scikit-learn para dividir los datos, escalarlos y medir qué tan bien predice el modelo

In [11]:
import warnings
# Ignoramos algunos warnings que se producen por invocar el pipeline sin el nombre de las características
warnings.filterwarnings('ignore', category=UserWarning, message='.*X does not have valid feature names.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*invalid value encountered in subtract.*')

**Interpretacion:**
* importamos los warnings para limpiar la pantalla y evitar que salgan alertas técnicas molestas cuando grafiquemos los resultados, ya que Python suele lanzar avisos que no afectan el funcionamiento

## Funciones auxiliares

In [12]:
# Construcción de una función que realice el particionado completo
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

**Interpretacion:**
* Creamos una función propia para partir nuestra tabla de datos en tres grupos: 60% para entrenar el modelo, 20% para validarlo y 20% para la prueba final. Usamos random_state=42 para que siempre se divida igual cada vez que ejecutemos el código

In [13]:
# Representación gráfica del límite de decisión
def plot_svc_decision_boundary(svm_clf, xmin, xmax):
    w = svm_clf.coef_[0]
    b = svm_clf.intercept_[0]

    # At the decision boundary, w0*x0 + w1*x1 + b = 0
    # => x1 = -w0/w1 * x0 - b/w1
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0]/w[1] * x0 - b/w[1]

    margin = 1/w[1]
    gutter_up = decision_boundary + margin
    gutter_down = decision_boundary - margin

    svs = svm_clf.support_vectors_
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#FFAAAA')
    plt.plot(x0, decision_boundary, "k-", linewidth=2)
    plt.plot(x0, gutter_up, "k--", linewidth=2)
    plt.plot(x0, gutter_down, "k--", linewidth=2)

**Interpretacion:**
* Esta función nos ayuda a visualizar cómo el SVM lineal separa los datos en un gráfico 2D. Calcula y dibuja la línea central que divide ambas clases, las dos líneas punteadas que representan el margen de seguridad y resalta con círculos rosados los vectores de soporte, que son los puntos clave que sostienen esa frontera.

## 1. Lectura del conjunto de datos

In [15]:
df = pd.read_csv("Phishing.csv")

* Leemos el archivo CSV con los datos de las URLs y lo guardamos en una tabla llamada df para empezar a trabajar con el

## 2. Visualización preliminar de la información

In [ ]:
df.head(10)

* Muestra las primeras 10 filas para ver cómo lucen los datos

In [ ]:
df.describe()

* Nos da promedios, mínimos y máximos y aquí notamos que las columnas tienen escalas muy distintas (unas van de 0 a 1 y otras llegan a 173)

In [ ]:
df.info()

* Confirma que tenemos 15,367 filas, 80 columnas y nos deja ver si faltan datos

In [ ]:
df["URL_Type_obf_Type"].value_counts()

* Cuenta cuántas URLs son seguras (benign: 7,781) y cuántas son maliciosas (phishing: 7,586), demostrando que las clases están bien equilibradas

In [ ]:
# Comprobación de si existen valores nulos
is_null = df.isna().any()
is_null[is_null]

* Revisamos si hay celdas vacías o errores matemáticos (infinito), porque los algoritmos de Machine Learning fallan si encuentran esto. Descubrimos 9 columnas con datos faltantes y 1 columna con valores infinitos.

In [ ]:
# Comprobación de la existencia de valores infinitos
is_inf = df.isin([np.inf, -np.inf]).any()
is_inf[is_inf]

In [ ]:
# Representación gráfica de dos características
plt.figure(figsize=(12, 6))
plt.scatter(df["domainUrlRatio"][df['URL_Type_obf_Type'] == "phishing"], df["domainlength"][df['URL_Type_obf_Type'] == "phishing"], c="r", marker=".")
plt.scatter(df["domainUrlRatio"][df['URL_Type_obf_Type'] == "benign"], df["domainlength"][df['URL_Type_obf_Type'] == "benign"], c="g", marker="x")
plt.xlabel("domainUrlRatio", fontsize=13)
plt.ylabel("domainlength", fontsize=13)
plt.show()

**Interpretacion:**
* Dibujamos un gráfico de dispersión usando solo dos características (domainUrlRatio y domainlength) para ver cómo se comportan las URLs
* Los puntos rojos son phishing y las cruces verdes son seguras, Vemos que están bastante mezclados, lo que nos indica que separar ambas clases con una simple línea recta no será tarea fácil

## 3. División del conjunto de datos

In [ ]:
# División del conjunto de datos
train_set, val_set, test_set = train_val_test_split(df)

In [ ]:
X_train = train_set.drop("URL_Type_obf_Type", axis=1)
y_train = train_set["URL_Type_obf_Type"].copy()

X_val = val_set.drop("URL_Type_obf_Type", axis=1)
y_val = val_set["URL_Type_obf_Type"].copy()

X_test = test_set.drop("URL_Type_obf_Type", axis=1)
y_test = test_set["URL_Type_obf_Type"].copy()

**Interpretacion**
* Aplicamos nuestra función para dividir los datos en entrenamiento, validación y prueba
* Luego, en cada grupo separamos las características (X) (las pistas que usará el modelo) de la etiqueta (y) (la respuesta correcta que queremos predecir: si es phishing o benigna).

## 4. Preparación del conjunto de datos

In [ ]:
# Eliminamos el atributo que tiene valores infinitos
X_train = X_train.drop("argPathRatio", axis=1)
X_val = X_val.drop("argPathRatio", axis=1)
X_test = X_test.drop("argPathRatio", axis=1)

**Limpiamos los errores detectados anteriormente:**
* Borramos la columna argPathRatio en los tres conjuntos porque tenía valores infinitos.

In [ ]:
# Rellenamos los valores nulos con la mediana
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

* Usamos SimpleImputer con la estrategia de la mediana para rellenar los huecos vacíos (NaN), ya que la mediana no se ve afectada por valores extremos

In [ ]:
# Rellenamos los valores nulos
X_train_prep = imputer.fit_transform(X_train)
X_val_prep = imputer.fit_transform(X_val)
X_test_prep = imputer.fit_transform(X_test)

In [ ]:
# Transformamos el resultado a un DataFrame de Pandas
X_train_prep = pd.DataFrame(X_train_prep, columns=X_train.columns, index=y_train.index)
X_val_prep = pd.DataFrame(X_val_prep, columns=X_val.columns, index=y_val.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=X_test.columns, index=y_test.index)

* Como SimpleImputer devuelve los datos en formato de matriz numérica sin encabezados, los volvemos a convertir en tablas (DataFrame) para recuperar los nombres de las columnas.

In [ ]:
X_train_prep.head(10)

In [ ]:
# Comprobamos si hay valores nulos en el conjunto de datos de entrenamiento
is_null = X_train_prep.isna().any()
is_null[is_null]

* Comprobamos que la tabla limpia se vea bien y volvemos a buscar valores nulos
* El resultado sale vacío (Series([], dtype: bool)), confirmando que ya no queda ningún hueco y los datos están listos para entrenar

## 5. SMV: Kernel lineal

### 5.1 Conjunto de datos reducido

**Entrenamiento del algoritmo con un conjunto de datos reducido**

In [ ]:
# Reducimos el conjunto de datos para representarlo gráficamente
X_train_reduced = X_train_prep[["domainUrlRatio", "domainlength"]].copy()
X_val_reduced = X_val_prep[["domainUrlRatio", "domainlength"]].copy()

In [ ]:
X_train_reduced

In [ ]:
from sklearn.svm import SVC

# SVM Large Margin Classification
svm_clf = SVC(kernel="linear", C=50)
svm_clf.fit(X_train_reduced, y_train)

**Interpretacion:**
* Para poder graficar cómo funciona el modelo en 2D, recortamos los datos quedándonos solo con 2 columnas (domainUrlRatio y domainlength)
* Luego creamos y entrenamos nuestro primer modelo SVM con kernel lineal (una línea recta) y un parámetro C=50, lo que obliga al modelo a ser estricto e intentar cometer la menor cantidad de errores posible al trazar la frontera

**Representación del límite de decisión**

In [ ]:
def plot_svc_decision_boundary(svm_clf, xmin, xmax):
    w = svm_clf.coef_[0]
    b = svm_clf.intercept_[0]

    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0]/w[1] * x0 - b/w[1]

    margin = 1/w[1]
    gutter_up = decision_boundary + margin
    gutter_down = decision_boundary - margin

    svs = svm_clf.support_vectors_
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#FFAAAA')
    plt.plot(x0, decision_boundary, "k-", linewidth=2)
    plt.plot(x0, gutter_up, "k--", linewidth=2)
    plt.plot(x0, gutter_down, "k--", linewidth=2)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(X_train_reduced.values[:, 0][y_train=="phishing"], X_train_reduced.values[:, 1][y_train=="phishing"], "g^")
plt.plot(X_train_reduced.values[:, 0][y_train=="benign"], X_train_reduced.values[:, 1][y_train=="benign"], "bs")
plot_svc_decision_boundary(svm_clf, 0, 1)
plt.title("$C = {}$".format(svm_clf.C), fontsize=16)
plt.axis([0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=13)
plt.ylabel("domainlength", fontsize=13)
plt.show()

**Interpretacion:**
* Dibujamos los datos de entrenamiento (triángulos verdes para phishing y cuadrados azules para benignas) y llamamos a nuestra función plot_svc_decision_boundary para superponer la línea negra que encontró el algoritmo. Con esto visualizamos exactamente dónde corta el modelo para decidir si una URL es maliciosa o no, y qué puntos quedaron dentro del margen de error

**Predicción con un conjunto de datos reducido**

In [ ]:
y_pred = svm_clf.predict(X_val_reduced)

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val, pos_label='phishing'))

Como se verá más adelante, para determinados kernels es muy importante escalar el conjunto de datos. En ese caso, para el kernel lineal, no es tan relevante, aunque es posible que proporciones mejores resultados.

In [ ]:
svm_clf_sc = Pipeline([
        ("scaler", RobustScaler()),
        ("linear_svc", SVC(kernel="linear", C=50)),
    ])

svm_clf_sc.fit(X_train_reduced, y_train)

* svm_clf_sc = Pipeline([...]): Crea un flujo que primero escala los datos con RobustScaler y luego aplica el modelo SVM lineal.
* svm_clf_sc.fit(...): Entrena todo el flujo con los datos reducidos

In [ ]:
y_pred = svm_clf_sc.predict(X_val_reduced)

* Predice las etiquetas de validación usando el modelo con escalado

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val, pos_label='phishing'))

* Imprime el puntaje F1 Score obtenido con el modelo escalado

### 5.2 Conjunto de datos completo

In [ ]:
# Entrenamiento con todo el conjunto de datos
from sklearn.svm import SVC

svm_clf = SVC(kernel="linear", C=1)
svm_clf.fit(X_train_prep, y_train)

* Carga nuevamente el clasificador SVM.
* Crea un SVM lineal con un parámetro de castigo C=1 usando todas las columnas.
* Entrena el modelo utilizando todo el conjunto de datos limpio.

In [ ]:
y_pred = svm_clf.predict(X_val_prep)

* Realiza las predicciones sobre todo el conjunto de validación limpio.

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val, pos_label='phishing'))

## 6. SMV: Kernel no lineal

### 6.1. Polynomial Kernel (I)

**Entrenamiento del algoritmo con un conjunto de datos reducido**

In [ ]:
# Para representar el límite de decisión tenemos que pasar la variable objetivo a numérica
y_train_num = y_train.factorize()[0]
y_val_num = y_val.factorize()[0]

* Convierte las etiquetas de texto de entrenamiento en números (0 y 1).
* y_val_num = y_val.factorize()[0]: Convierte las etiquetas de texto de validación en números.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import LinearSVC
from sklearn.preprocessing import PolynomialFeatures

polynomial_svm_clf = Pipeline([
        ("poly_features", PolynomialFeatures(degree=3)),
        ("scaler", StandardScaler()),
        ("svm_clf", LinearSVC(C=20, loss="hinge", random_state=42, max_iter=100000, dual=True))
    ])

polynomial_svm_clf.fit(X_train_reduced, y_train_num)

**Creamos un flujo (Pipeline) manual para hacer un SVM polinómico:**
* Primero elevamos los datos a grado 3 con PolynomialFeatures para capturar relaciones curvas, luego los estandarizamos, y finalmente entrenamos un clasificador lineal (LinearSVC) con un castigo estricto C=20

**Representación del límite de decisión**

In [ ]:
def plot_dataset(X, y):
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g.")
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "b.")

* Definimos una función rápida para dibujar en el gráfico los puntos reales de entrenamiento, separándolos por colores (verde para la clase 1 y azul para la clase 0)

In [ ]:
def plot_predictions(clf, axes):
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X).reshape(x0.shape)
    y_decision = clf.decision_function(X).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)
    plt.contourf(x0, x1, y_decision, cmap=plt.cm.brg, alpha=0.1)

fig, axes = plt.subplots(ncols=2, figsize=(15,5), sharey=True)
plt.sca(axes[0])
plot_dataset(X_train_reduced.values, y_train_num)
plot_predictions(polynomial_svm_clf, [0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=11)
plt.ylabel("domainlength", fontsize=11)
plt.sca(axes[1])
plot_predictions(polynomial_svm_clf, [0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=11)
plt.ylabel("domainlength", fontsize=11)
plt.show()

* Creamos una función para colorear el fondo del gráfico según lo que predice el modelo en cada zona. Luego generamos una ventana con dos paneles lado a lado para visualizar las fronteras de decisión curvas que logró trazar este modelo polinómic

**Predicción con el conjunto de datos reducido**

In [ ]:
y_pred = polynomial_svm_clf.predict(X_val_reduced)

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val_num))

* Usamos nuestro modelo polinómico manual para predecir las etiquetas del conjunto de validación reducido
* Imprimimos el puntaje F1 Score para evaluar qué tan acertadas fueron las predicciones de este modelo

### 6.2. Polynomial Kernel (II)

Existe una forma más sencilla de entrenar un algoritmo SVM que utilize polynomial kernel utilizando el parámetro **kernel** de la propia función implementada en sklearn

**Entrenamiento del algoritmo con un conjunto de datos reducido**

In [ ]:
svm_clf = SVC(kernel="poly", degree=3, coef0=10, C=20)
svm_clf.fit(X_train_reduced, y_train_num)

**Aquí usamos una forma mucho más directa y sencilla que ofrece Scikit-Learn:**
* Creamos un SVM configurando directamente el parámetro kernel="poly" (grado 3, sesgo coef0=10 y C=20), y lo entrenamos con los datos reducidos

**Representación del límite de decisión**

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(15,5), sharey=True)
plt.sca(axes[0])
plot_dataset(X_train_reduced.values, y_train_num)
plot_predictions(svm_clf, [0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=11)
plt.ylabel("domainlength", fontsize=11)
plt.sca(axes[1])
plot_predictions(svm_clf, [0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=11)
plt.ylabel("domainlength", fontsize=11)
plt.show()

* Dibujamos nuevamente los dos gráficos paralelos para ver las nuevas fronteras curvas que genera este kernel polinómico integrado sobre los datos de entrenamiento

**Predicción con un conjunto de datos reducido**

In [ ]:
y_pred = svm_clf.predict(X_val_reduced)

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val_num))

* Predecimos las etiquetas de validación utilizando este modelo de kernel polinómico directo
* Imprimimos el F1 Score obtenido para verificar su rendimiento con los datos reducidos.

**Predicción con el conjunto de datos completo**

In [ ]:
svm_clf = SVC(kernel="poly", degree=3, coef0=10, C=40)
svm_clf.fit(X_train_prep, y_train_num)

* Entrenamos de nuevo el kernel polinómico pero ahora usando todo el conjunto de datos completo y limpio (X_train_prep), aumentando además el valor de C a 40 para ser más estrictos con los errores

In [ ]:
y_pred = svm_clf.predict(X_val_prep)

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val_num))

* Generamos las predicciones sobre el conjunto de validación completo
* Imprimimos el puntaje F1 Score final, comprobando cómo mejora el modelo al entrenarse con todas las columnas disponibles

### 6.2. Gaussian Kernel

**Entrenamiento del algoritmo con un conjunto de datos reducido**

In [ ]:
rbf_kernel_svm_clf = Pipeline([
            ("scaler", RobustScaler()),
            ("svm_clf", SVC(kernel="rbf", gamma=0.5, C=1000))
        ])

rbf_kernel_svm_clf.fit(X_train_reduced, y_train_num)

* Creamos un flujo que primero escala los datos con RobustScaler (para evitar problemas con valores atípicos) y luego entrena un SVM con kernel Gaussiano o RBF (gamma=0.5, C=1000), usando los datos reducidos

**Representación del límite de decisión**

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(15,5), sharey=True)
plt.sca(axes[0])
plot_dataset(X_train_reduced.values, y_train_num)
plot_predictions(rbf_kernel_svm_clf, [0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=11)
plt.ylabel("domainlength", fontsize=11)
plt.sca(axes[1])
plot_predictions(rbf_kernel_svm_clf, [0, 1, -100, 250])
plt.xlabel("domainUrlRatio", fontsize=11)
plt.ylabel("domainlength", fontsize=11)
plt.show()

* Visualizamos en los dos gráficos cómo el kernel Gaussiano es capaz de crear islas y fronteras de decisión mucho más complejas y flexibles alrededor de los datos

**Predicción con un conjunto de datos reducido**

In [ ]:
y_pred = rbf_kernel_svm_clf.predict(X_val_reduced)

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val_num))

* Realizamos las predicciones sobre el conjunto de validación reducido usando el modelo RBF
* Imprimimos el F1 Score obtenido con esta configuración reducida

**Predicción con un conjunto de datos completo**

In [ ]:
rbf_kernel_svm_clf = Pipeline([
            ("scaler", RobustScaler()),
            ("svm_clf", SVC(kernel="rbf", gamma=0.05, C=1000))
        ])

rbf_kernel_svm_clf.fit(X_train_prep, y_train_num)

* Entrenamos el modelo RBF final utilizando todo el conjunto de datos completo y limpio, ajustando el parámetro gamma a un valor más suave de 0.05 para evitar sobreajuste y mantener un C=1000

In [ ]:
y_pred = rbf_kernel_svm_clf.predict(X_val_prep)

In [ ]:
print("F1 Score:", f1_score(y_pred, y_val_num))

* Predecimos las etiquetas del conjunto de validación completo con este último modelo optimizado
* Imprimimos el puntaje F1 Score final del notebook, cerrando la evaluación completa del modelo con el kernel Gaussiano y todos los datos